# Find & Fix Quality Issues

## Systematic workflow for discovering problems, creating solutions, and deploying improvements safely

Learn to systematically find and fix quality issues in your GenAI application using a real-world scenario. In this hands-on workflow, we'll address accuracy problems where the BGP banking assistant is "hallucinating" financial information not present in the tool call data.

## The Problem Scenario

Your BGP banking assistant has been running in production, but customers report accuracy issues. Some responses contain fabricated details about:

- Account balances not matching the data returned by tool calls
- Transaction amounts that were invented or incorrect
- Loan terms and credit card details not supported by the tool data

## The Solution Workflow

Follow this systematic 6-step process to identify, fix, and safely deploy improvements:

1. **Discover quality issues** Find problematic traces using evaluation results
2. **Create Evaluation Datasets** Turn bad traces into targeted evaluation sets, good traces into regression sets
3. **Iterate on changes** Use MLflow Prompt Registry to track your changes
4. **Evaluate changes improved quality** Test that your changes addressed the quality problems
5. **Verify changes didn't cause a regression** Ensure fixes don't break user inputs that already work well
6. **Deploy**

This approach ensures evidence-based improvements rather than guesswork, with quantitative validation and safe deployment practices.

![test-new-prompt](https://i.imgur.com/4wlhT63.gif)

This notebook provides hands-on experience with MLflow's complete quality improvement workflow for GenAI applications. You'll learn to systematically identify quality issues in production traces, create evaluation datasets, test improved prompts, and make data-driven deployment decisions.


## Install packages (only required if running in a Databricks Notebook)

In [ ]:
%pip install -U -r ../requirements.txt
dbutils.library.restartPython()

## Environment Setup

Load environment variables and verify MLflow configuration.


In [ ]:
import sys
sys.path.append('../')
sys.path.append('../../')

import os
import mlflow
from mlflow_demo.utils import *

# Configure for the current participant: derives the object prefix, prompt name,
# model service, and experiment from YOUR username. The widgets only carry the
# shared catalog/schema/warehouse.
setup_databricks_notebook_env()

print('=== Environment Setup ===')
print(f'DATABRICKS_HOST: {os.getenv("DATABRICKS_HOST")}')
print(f'MLFLOW_EXPERIMENT_ID: {os.getenv("MLFLOW_EXPERIMENT_ID")}')
print(f'LLM_MODEL: {os.getenv("LLM_MODEL")}')
print(f'UC_CATALOG: {os.getenv("UC_CATALOG")}')
print(f'UC_SCHEMA: {os.getenv("UC_SCHEMA")}')
print(f'MLFLOW_TRACING_SQL_WAREHOUSE_ID: {os.getenv("MLFLOW_TRACING_SQL_WAREHOUSE_ID")}')
print('Environment variables loaded successfully!')

import logging
logging.getLogger('urllib3').setLevel(logging.ERROR)
logging.getLogger('mlflow').setLevel(logging.ERROR)

In [ ]:
from mlflow_demo.utils import generate_trace_links, generate_dataset_link, generate_prompt_link, generate_evaluation_comparison_link, generate_evaluation_links

# Step 1: Discover Quality Issues in Production Traces

Let's start by examining real production traces to identify quality problems. Once you have tracing in place and judges defined, you can systematically improve your GenAI application by analyzing where it's failing.

Your BGP banking assistant has been running in production, but customers report accuracy issues. Some responses contain fabricated details about:

- Account balances not matching the data returned by tool calls
- Transaction amounts that were invented or incorrect
- Loan terms and credit card details not supported by the tool data

Here, we will:

1. Use the `accuracy` scorer to identify problematic traces we need to fix
2. Use the `accuracy`, `relevance`, and `actionability` scorers to identify traces that are performing well so we can ensure our change doesn't regress quality

**IMPORTANT**: In this notebook, we show you how to find traces using the MLflow SDKs. You can also perform these same steps using the MLflow Experiment UI.

**Documentation**

- [Build Evaluation Datasets](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset) - Create datasets from production traces
- [Search and Filter Traces](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/manage-traces) - Find specific traces by criteria

Run the next cells to search for traces with quality issues!


In [ ]:
import mlflow, pandas as pd
print('Searching for production traces...')
traces = mlflow.search_traces(max_results=50, filter_string='tags.sample_data = "yes"')
print(f'Found {len(traces)} total traces')
failed_accuracy_traces_data, high_quality_traces_data = [], []
print('\nAnalyzing trace assessments...')
for idx, trace_row in traces.iterrows():
  assessments = trace_row.assessments
  trace_id = getattr(trace_row.info, 'trace_id', None)
  if not trace_id and 'trace_id' in trace_row: trace_id = trace_row['trace_id']
  if not assessments: continue
  passed, failed_acc = 0, False
  for a in assessments:
    name, fb = a.get('assessment_name'), a.get('feedback', {}).get('value')
    if name == 'accuracy' and fb == 'no': failed_acc = True
    elif name in ['relevance', 'actionability', 'accuracy'] and fb == 'yes': passed += 1
  if failed_acc and len(failed_accuracy_traces_data) < 5:
    failed_accuracy_traces_data.append(trace_row)
    _, url = generate_trace_links(trace_id, print_urls=False)
    print(f'Failed: {trace_id[:8]}... (view: {url})')
  elif passed >= 3 and len(high_quality_traces_data) < 5:
    high_quality_traces_data.append(trace_row)
    print(f'High quality: {trace_id[:8]}... (passed {passed})')

In [ ]:
failed_accuracy_traces = pd.DataFrame(failed_accuracy_traces_data)
high_quality_traces = pd.DataFrame(high_quality_traces_data)
print(f'\nSaved {len(failed_accuracy_traces)} failed and {len(high_quality_traces)} high-quality traces')

# Step 2: Create Evaluation and Regression Datasets

Now that we've identified problematic traces, let's create two Evaluation Datasets. An MLflow Evaluation Dataset is an MLflow primitive that allows you to curate collections of traces to use for evaluation. Evaluation Datasets are stored as Delta Tables in Unity Catalog.

1. **Dataset with problematic traces**: Contains traces with accuracy issues (to test our improvements)
2. **Regression dataset with high-quality traces**: Contains high-quality traces (to ensure we don't break existing functionality)

**Documentation**

- [Create Evaluation Datasets](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset) - Build datasets from traces

Run the next cells to create these datasets!


In [ ]:
from mlflow.genai import datasets
from datetime import datetime

UC_CATALOG = os.getenv('UC_CATALOG', 'default_catalog')
UC_SCHEMA = os.getenv('UC_SCHEMA', 'default_schema')

LOW_ACCURACY_DATASET_NAME = f'low_accuracy_{datetime.now().strftime("%Y%m%d%H%M%S")}'
REGRESSION_DATASET_NAME = f'regression_set_{datetime.now().strftime("%Y%m%d%H%M%S")}'

print(f'\nCreating datasets in {UC_CATALOG}.{UC_SCHEMA}...')

low_accuracy_dataset = datasets.create_dataset(
    uc_table_name=f'{UC_CATALOG}.{UC_SCHEMA}.{LOW_ACCURACY_DATASET_NAME}',
)
print(f'Created low accuracy dataset: {LOW_ACCURACY_DATASET_NAME}')
low_accuracy_dataset.merge_records(failed_accuracy_traces)
print(f'Added {len(failed_accuracy_traces)} records into low accuracy dataset')
generate_dataset_link(low_accuracy_dataset.dataset_id)

regression_dataset = datasets.create_dataset(
    uc_table_name=f'{UC_CATALOG}.{UC_SCHEMA}.{REGRESSION_DATASET_NAME}',
)
print(f'Created regression dataset: {REGRESSION_DATASET_NAME}')
regression_dataset.merge_records(high_quality_traces)
print(f'Added {len(high_quality_traces)} records into regression dataset')
generate_dataset_link(regression_dataset.dataset_id)

# Step 3: Investigate Quality Issues: Understanding What's Going Wrong

Now let's dive deeper into the problematic traces to understand the root causes of our quality issues. By analyzing the judge rationales and specific examples, we can identify patterns in the failures and design targeted improvements.

## What We'll Investigate

**Judge Rationales Analysis**: Examine why the accuracy scorer flagged these traces as problematic

- What specific fabrications or hallucinations occurred?
- Which types of information are being invented (balances, transactions, loans)?
- Are there patterns in the failure modes?

**Root Cause Analysis**: Determine the underlying prompt issues

- Is the current prompt too vague about data constraints?
- Are there missing guardrails against fabrication?
- Does the prompt encourage guessing over acknowledging missing data?

Run the next cell to examine specific failures in the MLflow UI!


In [ ]:
print('Investigating Failed Accuracy Traces:')
print('=' * 80)

for idx, failed_trace in failed_accuracy_traces.iterrows():
    trace_info = failed_trace.info
    trace_id = getattr(trace_info, 'trace_id', None)
    _, trace_url = generate_trace_links(trace_id, print_urls=False)
    print(f'\nTrace ID: {trace_id[:8] if trace_id else "Unknown"}...')
    print(f'   View Trace: {trace_url}')

    for assessment in failed_trace.assessments:
        name = assessment.get('assessment_name')
        rationale = assessment.get('rationale')
        feedback = assessment.get('feedback', {}).get('value')
        if name == 'accuracy' and feedback == 'no':
            import textwrap
            wrapped = textwrap.fill(rationale, width=80,
                initial_indent='      ', subsequent_indent='      ')
            print(f'   Accuracy Judge Rationale:')
            print(wrapped)
            break

print('\nPatterns we noticed:')
print('   Fabricated account balance not in tool results')
print('   Invented transaction amounts or merchant names')
print('   Hallucinated loan terms or interest rates')
print('   Credit card details not matching tool call data')
print('\nThese patterns will inform our prompt improvements in Step 4!')
print('=' * 80)

# Step 4: Develop and Test Improved Prompts

Now that we understand the problem (hallucinated financial information) and have datasets to test against, let's create an improved prompt. We'll use MLflow Prompt Registry to manage prompt versions and enable safe iteration.

Based on our root cause analysis, the main issue is that the current prompt doesn't explicitly prevent fabrication. Our improved prompt will include:

- **Explicit NO FABRICATION rules** to prevent hallucinations
- **Clear data-only requirements** for all financial references
- **Enhanced accuracy guidelines** with specific banking failure conditions

**Documentation**

- [Create and Edit Prompts](https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry/create-and-edit-prompts) - Prompt Registry management

Run the next cells to create and compare prompt versions!


In [ ]:
import mlflow
from mlflow_demo.agent.prompts import FIXED_PROMPT_TEMPLATE

UC_SCHEMA = os.getenv('UC_SCHEMA')
UC_CATALOG = os.getenv('UC_CATALOG')
PROMPT_NAME = os.getenv('PROMPT_NAME')

print('Creating new prompt version in MLflow Prompt Registry...')

new_prompt = mlflow.genai.register_prompt(
    name=f'{UC_CATALOG}.{UC_SCHEMA}.{PROMPT_NAME}',
    template=FIXED_PROMPT_TEMPLATE,
    commit_message='Improved banking prompt to fix accuracy and hallucination issues.',
)

mlflow.genai.set_prompt_alias(
    name=f'{UC_CATALOG}.{UC_SCHEMA}.{PROMPT_NAME}',
    alias='test_prompt',
    version=new_prompt.version,
)

print(f'Created new prompt version: {new_prompt.name} (version {new_prompt.version})')
generate_prompt_link(f'{UC_CATALOG}.{UC_SCHEMA}.{PROMPT_NAME}')

# Step 5: Evaluate Improved Prompt Against Problem Cases

Now let's test our improved prompt against the evaluation dataset containing problematic traces. This will show us whether our changes actually fix the accuracy issues we identified.

We'll run evaluations with the same scorers used in production to ensure consistent quality measurement:

- **Accuracy**: Verifies all financial figures match tool call data
- **Relevance**: Ensuring content remains relevant to customer needs
- **Actionability**: Providing actionable banking guidance
- **Safety**: Basic safety checks

**Documentation**

- [Evaluate Apps](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/evaluate-app) - Run evaluations on datasets

Run the next cell to evaluate the improved prompt against both datasets!


In [ ]:
from mlflow_demo.evaluation.evaluator import SCORERS
from mlflow_demo.agent.agent import AGENT
from mlflow_demo.agent import agent as agent_module
from mlflow.types.responses import Message, ResponsesAgentRequest

def predict_fn(input):
    msgs = [m if isinstance(m, Message) else Message(**m) for m in input]
    request = ResponsesAgentRequest(input=msgs)
    return AGENT.predict(request)

print('Running evaluation of the old prompt...')
eval_results_old = mlflow.genai.evaluate(
    data=low_accuracy_dataset, predict_fn=predict_fn, scorers=SCORERS,
)
print('Old prompt evaluation completed!')

print('Switching to new prompt...')
agent_module.SYSTEM_PROMPT = mlflow.genai.load_prompt(
    f'prompts:/{UC_CATALOG}.{UC_SCHEMA}.{PROMPT_NAME}@test_prompt')

print('Running evaluation of the new prompt...')
eval_results_new = mlflow.genai.evaluate(
    data=low_accuracy_dataset, predict_fn=predict_fn, scorers=SCORERS,
)
print('New prompt evaluation completed!')
print('=' * 60)
print('View the results in the MLflow comparison UI:')
generate_evaluation_comparison_link(eval_results_new.run_id, eval_results_old.run_id)

## Now, you can keep iterating on the prompt using the evaluation results and scorer outputs to guide your improvements!

# Step 6: Verify No Regressions on High-Quality Examples

Now we need to ensure our improvements don't break existing functionality. We'll run the improved prompt against our regression dataset containing high-quality traces to verify that:

- Existing good examples remain good
- Quality metrics don't decrease for well-performing cases
- New accuracy rules don't interfere with proper responses

This step is crucial for safe deployment - we want to fix problems without creating new ones.

Run the next cell to verify no regressions on high-quality examples!


In [ ]:
print('Running regression evaluation of the new prompt...')

regression_results = mlflow.genai.evaluate(
    data=regression_dataset, predict_fn=predict_fn, scorers=SCORERS,
)

generate_evaluation_links(regression_results.run_id)

# Exercise 1

Modify the FIXED_PROMPT_TEMPLATE to add a rule: 'Always include the account type when discussing balances.' Register it and evaluate.


# Exercise 2

Compare the evaluation results. Would you deploy your version? Why or why not?


# Exercise 3

Create a third 'borderline' dataset with traces that passed 2 out of 3 scorers and test against it.


# Step 7: Deploy the new version

Based on our evaluation results, we can now make an informed decision about additional improvements to reach our quality bar OR feel confident deploying the prompt knowing quality will improve without any regressions!


# Tutorial Complete: Systematic GenAI Quality Improvement

Congratulations! You've successfully completed the end-to-end workflow for systematically improving GenAI quality using MLflow's evaluation and prompt management capabilities.

## What You've Accomplished

- **Identified Quality Issues** - Found traces with accuracy problems through systematic analysis
- **Created Targeted Datasets** - Built evaluation datasets from problematic traces and regression datasets from high-quality examples
- **Developed Improved Prompts** - Created enhanced prompts with explicit accuracy requirements using MLflow Prompt Registry
- **Validated Improvements** - Tested improved prompts against problem cases and verified no regressions

## The Complete MLflow Quality Improvement Workflow

1. **Discover quality issues** Find problematic traces using evaluation results
2. **Create Evaluation Datasets** Turn bad traces into targeted evaluation sets, good traces into regression sets
3. **Iterate on changes** Use MLflow Prompt Registry to track your changes
4. **Evaluate changes improved quality** Test that your changes addressed the quality problems
5. **Verify changes didn't cause a regression** Ensure fixes don't break user inputs that already work well
6. **Deploy**
